In [ ]:
import glob, os, shutil, subprocess, sys

REPO = "https://github.com/yanis-ghazi/pedwri-fracture"
local = glob.glob("/kaggle/input/**/pedwri-fracture/pyproject.toml", recursive=True)
if local:
    src = shutil.copytree(os.path.dirname(local[0]), "/kaggle/working/pedwri-src", dirs_exist_ok=True)
    target = f"{src}[train]"
else:
    target = f"pedwri[train] @ git+{REPO}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], check=True)

In [ ]:
PREPARED = glob.glob("/kaggle/input/**/prepared/info.json", recursive=True)[0].rsplit("/", 1)[0]
WORK = "/kaggle/working"
!nvidia-smi --query-gpu=name,memory.total --format=csv
WEIGHTS = f"{WORK}/runs/yolo11s_1024/weights/best.pt"

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
src, dst = Path(PREPARED), Path("/tmp/prepared")
files = [p for p in src.rglob("*") if p.is_file()]
for d in {dst / p.parent.relative_to(src) for p in files}: d.mkdir(parents=True, exist_ok=True)
with ThreadPoolExecutor(32) as ex: list(ex.map(lambda p: shutil.copyfile(p, dst / p.relative_to(src)), files))
PREPARED = str(dst)

In [ ]:
!python -m pedwri.train --data {PREPARED} --project {WORK}/runs --name yolo11s_1024 --epochs 60 --batch 16

In [ ]:
!python -m pedwri.predict --weights {WEIGHTS} --data {PREPARED} --out {WORK}/predictions

In [ ]:
!python -m pedwri.evaluate --data {PREPARED} --pred-dir {WORK}/predictions --out {WORK}/results

In [ ]:
!python -m pedwri.errors --results {WORK}/results --images {PREPARED}/images/test

In [ ]:
!python -m pedwri.export --weights {WEIGHTS} --out {WORK}/models
!cp {WORK}/results/operating_points.json {WORK}/models/
!ls -lh {WORK}/models